In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('/projects/activities/kappsen-tmc/USERS/domans/differential-annotator-dev/image-differential-annotator/')

from annotator.annotation import runAnnotation
from annotator.combineCDF import getDiscreteCombinedCDFofAllFeatures as PCMA
from annotator.stqutils import loadAdFromH5, preparePatchesWSI, getPatchRepresentation, inferProb, showProb

import numpy as np
import pandas as pd

qs = np.linspace(0.05, 0.95, 10, endpoint=True)

In [ ]:
samples  = ['SA_001_L_C_a', 'SA_001_L_CM_w']

In [ ]:
ads = {}
imgs = {}

In [ ]:
# Load the pre-processed AnnData and image for each sample
for sample in samples:
    ads[sample], imgs[sample] = loadAdFromH5(f'/projects/activities/kappsen-tmc/USERS/domans/results-kidney-001/{sample}/', fname='img.data.ctranspath-1.h5ad')

In [ ]:
# Prepare the patches coordinates for each sample and concatenate them into a single DataFrame
patchCoordinates = pd.concat([preparePatchesWSI(ads[sample].obs, N=12, spacing=28/0.25, sample_id=sample) for sample in samples], axis=0)

# Get the patch SAMPLER representations for each sample and combine them into a single DataFrame
patchesCDFs = pd.concat([getPatchRepresentation(ads[sample], patchCoordinates.xs(sample, level='sample', axis=0), qs, sample_id=sample) for sample in samples], axis=0)

In [ ]:
clf, plog, bp = {}, [], {}
runAnnotation(patchCoordinates, patchesCDFs, imgs, bp, clf, plog, figsize=(5,5), augFunc=PCMA, alpha=0.5, seed=42, randomness=0.5)

In [ ]:
x, y, p = inferProb(ads['SA_001_L_C_a'], clf['clf'], qs, Nmax=5*10**5) # 1e5

In [ ]:
showProb(x, y, p, s=1, marker='s', figsize=(12, 12), colorbar=True, vmin=0.5, vmax=1)

In [ ]:
x, y, p = inferProb(ads['SA_001_L_CM_w'], clf['clf'], qs, Nmax=100*10**5)

In [ ]:
showProb(x, y, p, s=1, marker='s', figsize=(24, 12), colorbar=True, vmin=0.25, vmax=1)